# 03 — Self-Healing Pipeline Demo

This notebook demonstrates the full autonomous self-healing loop:
1. Check current model state
2. Trigger the Airflow self-healing DAG
3. Monitor task execution in real-time
4. Verify new model registered

**Prerequisites**: Docker services + Airflow running

In [1]:
import time

import httpx

API_URL = "http://localhost:8001"
AIRFLOW_URL = "http://localhost:8080"
AIRFLOW_AUTH = ("admin", "admin")

## 1. Current Model State

In [2]:
resp = httpx.get(f"{API_URL}/models/credit-risk")
model = resp.json()
print(f"Current Champion: v{model['version']}")
print(f"Status: {model['status']}")
print(f"Metadata: {model.get('metadata', {})}")

Current Champion: vv1
Status: development
Metadata: {'role': 'champion', 'metrics': {'accuracy': 0.85}}


## 2. Trigger Self-Healing Pipeline

Simulates drift detection triggering the Airflow DAG.

In [3]:
# Unpause the DAG first
httpx.patch(
    f"{AIRFLOW_URL}/api/v1/dags/self_healing_pipeline",
    json={"is_paused": False},
    auth=AIRFLOW_AUTH,
)

# Trigger with drift context
resp = httpx.post(
    f"{AIRFLOW_URL}/api/v1/dags/self_healing_pipeline/dagRuns",
    json={
        "conf": {
            "model_id": "credit-risk",
            "reason": "Notebook demo — simulated drift",
            "drift_score": 0.85,
        }
    },
    auth=AIRFLOW_AUTH,
)
run = resp.json()
dag_run_id = run["dag_run_id"]
print(f"DAG Run triggered: {dag_run_id}")
print(f"State: {run['state']}")

DAG Run triggered: manual__2026-03-19T17:37:48.456757+00:00
State: queued


## 3. Monitor Task Execution

In [4]:
import urllib.parse

encoded_run_id = urllib.parse.quote(dag_run_id, safe="")
tasks_url = f"{AIRFLOW_URL}/api/v1/dags/self_healing_pipeline/dagRuns/{encoded_run_id}/taskInstances"

EXPECTED_TASKS = {"send_alert", "rollback_challenger", "train_model", "log_mlflow", "register_model"}

for attempt in range(30):
    resp = httpx.get(tasks_url, auth=AIRFLOW_AUTH)
    tasks = resp.json().get("task_instances", [])

    completed = sum(1 for t in tasks if t["state"] == "success")
    failed = sum(1 for t in tasks if t["state"] == "failed")

    print(f"\r[{attempt+1}/30] Tasks: {completed}/5 complete, {failed} failed", end="")

    if completed == 5 or failed > 0:
        print()
        break
    time.sleep(5)

print("\n--- Task Results ---")
for t in sorted(tasks, key=lambda x: x.get("start_date") or ""):
    dur = f"({t['duration']:.1f}s)" if t.get("duration") else ""
    icon = "\u2705" if t["state"] == "success" else "\u274c"
    print(f"{icon} {t['task_id']:25s} | {t['state']:12s} {dur}")

[5/30] Tasks: 5/5 complete, 0 failed

--- Task Results ---
✅ send_alert                | success      (0.2s)
✅ rollback_challenger       | success      (0.2s)
✅ train_model               | success      (12.8s)
✅ log_mlflow                | success      (1.7s)
✅ register_model            | success      (0.3s)


## 4. Verify New Model Registered

In [5]:
resp = httpx.get(f"{API_URL}/models/credit-risk")
model = resp.json()
print("Model after self-healing:")
print(f"  Version: {model['version']}")
print(f"  Status:  {model['status']}")
print(f"  Metadata: {model.get('metadata', {})}")

Model after self-healing:
  Version: v1
  Status:  development
  Metadata: {'role': 'champion', 'metrics': {'accuracy': 0.85}}
